# EasyOCR Fine-Tuning for TesseractApexOCR

This notebook fine-tunes the EasyOCR English recognition model (`english_g2`) on the specific Apex Legends UI font to eliminate typos.

## 1. Setup Environment

In [ ]:
!git clone https://github.com/clovaai/deep-text-recognition-benchmark.git
%cd deep-text-recognition-benchmark
!pip install lmdb fire tensorboard

## 2. Upload Dataset
**Action Required**: Upload your `easyocr_lmdb_dataset.zip` file to the Colab workspace (inside `deep-text-recognition-benchmark`).

In [ ]:
import os
import zipfile

if not os.path.exists('easyocr_lmdb_dataset.zip'):
    print('ERROR: Please upload easyocr_lmdb_dataset.zip first!')
else:
    with zipfile.ZipFile('easyocr_lmdb_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print('Dataset extracted to dataset/')


## 3. Download Pre-trained Weights
We use the standard `english_g2` model from JaidedAI.

In [ ]:
!wget -O english_g2.pth https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.pth

## 4. Train the Model
EasyOCR's `english_g2` uses `None-VGG-BiLSTM-CTC`. We will fine-tune it for 10000 iterations.

In [ ]:
!python train.py \
  --train_data dataset/easyocr_dataset/train_lmdb \
  --valid_data dataset/easyocr_dataset/val_lmdb \
  --select_data "/" \
  --batch_ratio 1 \
  --Transformation None \
  --FeatureExtraction VGG \
  --SequenceModeling BiLSTM \
  --Prediction CTC \
  --saved_model english_g2.pth \
  --batch_size 128 \
  --data_filtering_off \
  --workers 4 \
  --keep_ratio \
  --num_iter 10000 \
  --valInterval 500 \
  --experiment_name apex_easyocr_finetune

## 5. Download Weights
After training completes, download the best model weights.

In [ ]:
from google.colab import files
files.download('saved_models/apex_easyocr_finetune/best_accuracy.pth')